# Practica 1 (30 min): GCN vs GAT vs GraphSAGE (SOLUCIONES)
## Master Oficial: Big Data Science

Notebook de referencia para rescate docente.
Incluye implementacion completa de los 3 TODOs de la version student.

In [ ]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATv2Conv, SAGEConv

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device} | Data dir: {DATA_DIR.resolve()}")

In [ ]:
def load_dataset_with_fallback(data_dir: Path):
    last_error = None
    for name in ["Cora", "CiteSeer"]:
        try:
            ds = Planetoid(root=str(data_dir / "planetoid"), name=name)
            return ds, ds[0], name
        except Exception as exc:
            last_error = exc
            print(f"Aviso: no se pudo cargar {name}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"No se pudo cargar Cora ni CiteSeer: {last_error}")

dataset, data, dataset_name = load_dataset_with_fallback(DATA_DIR)
data = data.to(device)

print(f"Dataset activo: {dataset_name}")
print(
    f"nodes={data.num_nodes} | edges={data.num_edges} | features={dataset.num_features} | classes={dataset.num_classes}"
)
print(
    f"train={int(data.train_mask.sum())} | val={int(data.val_mask.sum())} | test={int(data.test_mask.sum())}"
)

assert data.x.shape[0] == data.y.shape[0]
assert int(data.train_mask.sum()) > 0

In [ ]:
class GCNNet(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.conv1(h, edge_index)
        h = torch.relu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.conv2(h, edge_index)
        return logits, emb


class GATNet(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.gat2(h, edge_index)
        return logits, emb


class SAGENet(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.s1 = SAGEConv(in_dim, hidden_dim)
        self.s2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = self.s1(x, edge_index)
        h = F.relu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.s2(h, edge_index)
        return logits, emb

In [ ]:
def build_model(model_name, in_dim, out_dim, device, hidden_dim=None, heads=4):
    model_name = model_name.lower()
    if hidden_dim is None:
        hidden_dim = 16 if model_name == "gat" else 64

    if model_name == "gcn":
        model = GCNNet(in_dim, hidden_dim, out_dim)
    elif model_name == "gat":
        model = GATNet(in_dim, hidden_dim, out_dim, heads=heads)
    elif model_name == "sage":
        model = SAGENet(in_dim, hidden_dim, out_dim)
    else:
        raise ValueError(f"model_name invalido: {model_name}")

    return model.to(device)


m = build_model("gcn", dataset.num_features, dataset.num_classes, device)
m.eval()
with torch.no_grad():
    logits, emb = m(data.x, data.edge_index)
assert logits.shape == (data.num_nodes, dataset.num_classes)
assert emb.shape[0] == data.num_nodes
print("Checkpoint 1 OK")

In [ ]:
def accuracy(logits, y):
    pred = logits.argmax(dim=1)
    return float((pred == y).sum().item() / len(y))


def train_one_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())


@torch.no_grad()
def evaluate(model, data, criterion):
    model.eval()
    logits, _ = model(data.x, data.edge_index)

    val_loss = float(criterion(logits[data.val_mask], data.y[data.val_mask]).item())
    val_acc = accuracy(logits[data.val_mask], data.y[data.val_mask])
    test_acc = accuracy(logits[data.test_mask], data.y[data.test_mask])

    return val_loss, val_acc, test_acc

In [ ]:
def train_one_run(model_name, seed=42, hidden_dim=None, max_epochs=80, patience=15):
    seed_everything(seed)
    model = build_model(model_name, dataset.num_features, dataset.num_classes, device, hidden_dim=hidden_dim)

    lr = 0.01
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    best_state = None
    best_val_loss = float("inf")
    wait = 0

    t0 = time.perf_counter()
    for _ in range(max_epochs):
        _ = train_one_epoch(model, data, optimizer, criterion)
        val_loss, _, _ = evaluate(model, data, criterion)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    _, val_acc, test_acc = evaluate(model, data, criterion)
    elapsed = time.perf_counter() - t0

    return {
        "model": model_name,
        "seed": seed,
        "hidden_dim": hidden_dim if hidden_dim is not None else (16 if model_name == "gat" else 64),
        "val_acc": round(val_acc, 4),
        "test_acc": round(test_acc, 4),
        "time_sec": round(elapsed, 2),
    }


test_row = train_one_run("gcn", seed=42, max_epochs=20, patience=5)
assert 0.0 <= test_row["test_acc"] <= 1.0
assert test_row["time_sec"] >= 0.0
print("Checkpoint 2 OK")
print(test_row)

In [ ]:
def run_experiment(seed=42):
    rows = []
    for model_name in ["gcn", "gat", "sage"]:
        row = train_one_run(model_name, seed=seed, max_epochs=80, patience=15)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(by="test_acc", ascending=False).reset_index(drop=True)
    return df


results_df = run_experiment(seed=42)
assert len(results_df) == 3
print("Checkpoint 3 OK")
display(results_df)

In [ ]:
ABLATION_MODEL = "gat"
HIDDEN_DIMS = [8, 16, 32]

ablation_rows = []
for h in HIDDEN_DIMS:
    row = train_one_run(ABLATION_MODEL, seed=42, hidden_dim=h, max_epochs=60, patience=10)
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows).sort_values(by="hidden_dim").reset_index(drop=True)
display(ablation_df)

plt.figure(figsize=(6, 4))
plt.plot(ablation_df["hidden_dim"], ablation_df["test_acc"], marker="o")
plt.title(f"Ablation: {ABLATION_MODEL.upper()} hidden_dim vs test_acc")
plt.xlabel("hidden_dim")
plt.ylabel("test_acc")
plt.grid(alpha=0.3)
plt.show()

## Interpretacion sugerida

- Reporta que modelo lidera en accuracy y cual fue mas rapido.
- Comenta si el ablation mejora o degrada el rendimiento.
- Resume el trade-off observado entre calidad y coste.